# 🏥 ExplainableVLM-Rad: Google Colab Paper Reproduction Launcher

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vikhram-S/EXPLAINABLE-VLM-PAPER/blob/main/notebooks/01_colab_kaggle_launcher.ipynb)

This notebook provides the complete, end-to-end paper reproduction workflow for **ExplainableVLM-Rad**: An Explainable Vision-Language Model for Automated Chest Radiography Report Generation with Supervised Visual Phrase-Grounding.

---

## 1. Environment Setup & Repository Cloning

In [ ]:
import os
import sys

# Ensure base directory is /content in Colab to prevent nested cloning
if 'google.colab' in sys.modules:
    os.chdir('/content')

REPO_NAME = 'EXPLAINABLE-VLM-PAPER'
REPO_URL = 'https://github.com/Vikhram-S/EXPLAINABLE-VLM-PAPER.git'

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

# Set current working directory & python path
os.chdir(f'/content/{REPO_NAME}' if 'google.colab' in sys.modules else REPO_NAME)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Install required packages
!pip install -q -r requirements.txt
print(f"✅ Working Directory: {os.getcwd()}")
print("✅ Environment setup complete!")

## 2. Mount Google Drive for Persistent Storage & Check GPU

In [ ]:
import sys
import torch

# Mount Google Drive if in Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PERSISTENT_DIR = '/content/drive/MyDrive/explainablevlm_rad_outputs'
else:
    PERSISTENT_DIR = 'outputs'

os.makedirs(PERSISTENT_DIR, exist_ok=True)
print(f"📁 Output Directory: {PERSISTENT_DIR}")

# Verify GPU Hardware
print(f"⚡ PyTorch Version: {torch.__version__}")
print(f"⚡ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"🧠 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Data Pipeline Sanity Check

In [ ]:
# Verify dataset loader, patch grounding masks, and text preprocessing
!python scripts/sanity_check_data.py --num_samples 4

## 4. Execute Training Pipeline

- **Stage 1 (IU X-Ray Pre-training)**
- **Stage 2 (MIMIC-CXR Fine-tuning)**
- **Ablation Model (No Visual Grounding Loss)**

In [ ]:
# Stage 1: IU X-Ray Pre-training
!python src/train.py --config configs/experiments/stage1_iu_xray.yaml

# Stage 2: MIMIC-CXR Fine-tuning
!python src/train.py --config configs/experiments/stage2_mimic_cxr.yaml

# Ablation Study: No Grounding Loss
!python src/train.py --config configs/experiments/ablation_no_exp_loss.yaml

## 5. Master Evaluation Suite & Metrics Benchmark

Evaluates NLG (BLEU-1 to 4, ROUGE-L, CIDEr), Clinical F1 (CheXbert-14), and Visual Grounding IoU.

In [ ]:
!python src/eval/evaluator.py

## 6. Generate All 8 Journal-Quality Figures (300 DPI) & LaTeX Tables

In [ ]:
# Generate paper figures
!python src/viz/plot_curves.py
!python src/viz/plot_comparison.py
!python src/viz/plot_heatmaps.py
!python src/viz/plot_ablation.py
!python src/viz/plot_pathology.py
!python src/viz/plot_human_eval.py
!python src/viz/plot_faithfulness.py
!python src/viz/plot_qualitative_grid.py

# Generate camera-ready LaTeX tables
!python src/viz/generate_latex_tables.py

print("📊 All figures and LaTeX tables successfully generated in outputs/")

## 7. Manuscript Verification & Human Eval Sheet Generation

In [ ]:
# Text-data drift check between outputs/results_summary.json and manuscript draft
!python scripts/check_manuscript_consistency.py

# Generate randomized blinded evaluation sheet for board-certified radiologist scoring
!python scripts/generate_human_eval_sheet.py